In [ ]:
"""This module trains the diffusion model"""

import argparse
import os
import torch
import numpy as np
from torch import from_numpy, optim, nn, randint, normal, sqrt, device, save
from torch.utils.data import DataLoader
from data_utils import Preprocessor
from training_utils import MyDataset, fetchModel, fetchDiffusionConfig
from types import SimpleNamespace


CUDA extension for cauchy multiplication not found. Install by going to extensions/cauchy/ and running `python setup.py install`. This should speed up end-to-end training by 10-50%
Falling back on slow Cauchy kernel. Install at least one of pykeops or the CUDA extension for efficiency.


In [2]:
"""OLD Wavestitch"""

# np.random.seed(42)
# torch.manual_seed(42)
# parser = argparse.ArgumentParser()
# parser.add_argument('-dataset', '-d', type=str,
#                     help='MetroTraffic, BeijingAirQuality, AustraliaTourism, RossmanSales, PanamaEnergy', required=True)
# parser.add_argument('-backbone', type=str, help='Transformer, Bilinear, Linear, S4', default='S4')
# parser.add_argument('-beta_0', type=float, default=0.0001, help='initial variance schedule')
# parser.add_argument('-beta_T', type=float, default=0.02, help='last variance schedule')
# parser.add_argument('-timesteps', '-T', type=int, default=200, help='training/inference timesteps')
# parser.add_argument('-hdim', type=int, default=64, help='hidden embedding dimension')
# parser.add_argument('-lr', type=float, default=1e-4, help='learning rate')
# parser.add_argument('-batch_size', type=int, help='batch size', default=1024)
# parser.add_argument('-epochs', type=int, default=1000, help='training epochs')
# parser.add_argument('-layers', type=int, default=4, help='number of hidden layers')
# parser.add_argument('-window_size', type=int, default=32, help='the size of the training windows')
# parser.add_argument('-stride', type=int, default=1, help='the stride length to shift the training window by')
# parser.add_argument('-num_res_layers', type=int, default=4, help='the number of residual layers')
# parser.add_argument('-res_channels', type=int, default=64, help='the number of res channels')
# parser.add_argument('-skip_channels', type=int, default=64, help='the number of skip channels')
# parser.add_argument('-diff_step_embed_in', type=int, default=32, help='input embedding size diffusion')
# parser.add_argument('-diff_step_embed_mid', type=int, default=64, help='middle embedding size diffusion')
# parser.add_argument('-diff_step_embed_out', type=int, default=64, help='output embedding size diffusion')
# parser.add_argument('-s4_lmax', type=int, default=100)
# parser.add_argument('-s4_dstate', type=int, default=64)
# parser.add_argument('-s4_dropout', type=float, default=0.0)
# parser.add_argument('-s4_bidirectional', type=bool, default=True)
# parser.add_argument('-s4_layernorm', type=bool, default=True)
# parser.add_argument('-propCycEnc', type=bool, default=False)
# args = parser.parse_args()
# dataset = args.dataset
# device = device('cuda' if torch.cuda.is_available() else 'cpu')
# preprocessor = Preprocessor(dataset, args.propCycEnc)
# df = preprocessor.df_cleaned
# training_df = df.loc[preprocessor.train_indices]
# test_df = df.loc[preprocessor.test_indices]
# hierarchical_column_indices = training_df.columns.get_indexer(preprocessor.hierarchical_features_cyclic)
# # training_samples = []
# d_vals_tensor = from_numpy(training_df.values)
# training_samples = d_vals_tensor.unfold(0, args.window_size, 1).transpose(1, 2)
# # masks = m_vals_tensor.unfold(0, args.window_size, 1)
# # for i in range(0, len(training_df) - args.window_size + 1, args.stride):
# #     window = training_df.iloc[i:i + args.window_size].values
# #     training_samples.append(window)
# in_dim = len(training_df.columns)
# out_dim = len(training_df.columns) - len(hierarchical_column_indices)
# training_dataset = MyDataset(training_samples.float())
# model = fetchModel(in_dim, out_dim, args).to(device)
# diffusion_config = fetchDiffusionConfig(args)
# optimizer = optim.Adam(model.parameters(), lr=args.lr)
# criterion = nn.MSELoss()
# dataloader = DataLoader(training_dataset, batch_size=args.batch_size, shuffle=True)
# all_indices = np.arange(len(training_df.columns))

# # Find the indices not in the index_list
# remaining_indices = np.setdiff1d(all_indices, hierarchical_column_indices)

# # Convert to an ndarray
# non_hier_cols = np.array(remaining_indices)
# """TRAINING"""
# for epoch in range(args.epochs):
#     total_loss = 0.0
#     for batch in dataloader:
#         batch = batch.to(device)
#         timesteps = randint(diffusion_config['T'], size=(batch.shape[0],)).to(device)
#         sigmas = normal(0, 1, size=batch.shape).to(device)
#         """Forward noising"""
#         alpha_bars = diffusion_config['alpha_bars'].to(device)
#         coeff_1 = sqrt(alpha_bars[timesteps]).reshape((len(timesteps), 1, 1))
#         coeff_2 = sqrt(1 - alpha_bars[timesteps]).reshape((len(timesteps), 1, 1))
#         conditional_mask = np.ones(batch.shape)
#         conditional_mask[:, :, non_hier_cols] = 0
#         conditional_mask = from_numpy(conditional_mask).float().to(device)
#         batch_noised = (1 - conditional_mask) * (coeff_1 * batch + coeff_2 * sigmas) + conditional_mask * batch
#         batch_noised = batch_noised.to(device)
#         timesteps = timesteps.reshape((-1, 1))
#         # timesteps = timesteps.to(device)
#         sigmas_predicted = model(batch_noised, timesteps)
#         optimizer.zero_grad()
#         sigmas_permuted = sigmas[:, :, non_hier_cols].permute((0, 2, 1))
#         sigmas_permuted = sigmas_permuted.to(device)
#         loss = criterion(sigmas_predicted, sigmas_permuted)
#         loss.backward()
#         total_loss += loss
#         optimizer.step()
#     print(f'epoch: {epoch}, loss: {total_loss}')
# path = f'saved_models/{args.dataset}/'
# if args.propCycEnc:
#     filename = "model_prop.pth"
# else:
#     filename = "model.pth"
# filepath = os.path.join(path, filename)

# if not os.path.exists(path):
#     os.makedirs(path)
# torch.save(model.state_dict(), filepath)


'OLD Wavestitch'

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

# Hardcoded config
dataset            = 'RossmanSales'  # <- change this as needed
backbone           = 'S4'
beta_0             = 0.0001
beta_T             = 0.02
timesteps          = 200
hdim               = 64
lr                 = 1e-4
batch_size         = 1024
epochs             = 1000
layers             = 4
window_size        = 32
stride             = 1
num_res_layers     = 4
res_channels       = 64
skip_channels      = 64
diff_step_embed_in = 32
diff_step_embed_mid= 64
diff_step_embed_out= 64
s4_lmax            = 100
s4_dstate          = 64
s4_dropout         = 0.0
s4_bidirectional   = True
s4_layernorm       = True
propCycEnc         = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Preprocessing
preprocessor= Preprocessor(dataset, propCycEnc)
df          = preprocessor.df_cleaned
training_df = df.loc[preprocessor.train_indices]
test_df     = df.loc[preprocessor.test_indices]
hierarchical_column_indices = training_df.columns.get_indexer(preprocessor.hierarchical_features_cyclic)

# Data prep
d_vals_tensor   = from_numpy(training_df.values)
training_samples= d_vals_tensor.unfold(0, window_size, 1).transpose(1, 2)
in_dim          = len(training_df.columns)
out_dim         = len(training_df.columns) - len(hierarchical_column_indices)

# Dataset and model
training_dataset = MyDataset(training_samples.float())
args = {'backbone': backbone, 'beta_0': beta_0, 'beta_T': beta_T,
'timesteps': timesteps, 'hdim': hdim, 'lr': lr, 'batch_size': batch_size, 'epochs': epochs, 'layers': layers,
'window_size': window_size, 'stride': stride, 'num_res_layers': num_res_layers, 'res_channels': res_channels,
'skip_channels': skip_channels, 'diff_step_embed_in': diff_step_embed_in, 'diff_step_embed_mid': diff_step_embed_mid,
'diff_step_embed_out': diff_step_embed_out, 's4_lmax': s4_lmax, 's4_dstate': s4_dstate, 's4_dropout': s4_dropout,
's4_bidirectional': s4_bidirectional, 's4_layernorm': s4_layernorm, 'propCycEnc': propCycEnc}

args      = SimpleNamespace(**args)
model     = fetchModel(in_dim, out_dim, args).to(device)
diffusion_config = fetchDiffusionConfig(args)
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.MSELoss()
dataloader= DataLoader(training_dataset, batch_size=batch_size, shuffle=True)

# Get non-hierarchical columns
all_indices      = np.arange(len(training_df.columns))
remaining_indices= np.setdiff1d(all_indices, hierarchical_column_indices)
non_hier_cols    = np.array(remaining_indices)

# Training loop
for epoch in range(epochs):
    total_loss = 0.0
    for batch in dataloader:
        batch = batch.to(device)
        # t = randint(diffusion_config['T'], size=(batch.shape[0],)).to(device)
        t = torch.randint(diffusion_config['T'], (batch.shape[0],), device=device)
        # sigmas = normal(0, 1, size=batch.shape).to(device)
        sigmas    = torch.randn(batch.shape, device=device)
        alpha_bars= diffusion_config['alpha_bars'].to(device)
        coeff_1   = sqrt(alpha_bars[t]).reshape((len(t), 1, 1))
        coeff_2   = sqrt(1 - alpha_bars[t]).reshape((len(t), 1, 1))
        conditional_mask = np.ones(batch.shape)
        conditional_mask[:, :, non_hier_cols] = 0
        conditional_mask = from_numpy(conditional_mask).float().to(device)
        batch_noised = (1 - conditional_mask) * (coeff_1 * batch + coeff_2 * sigmas) + conditional_mask * batch
        batch_noised = batch_noised.to(device)
        t = t.reshape((-1, 1))
        sigmas_predicted = model(batch_noised, t)
        optimizer.zero_grad()
        sigmas_permuted = sigmas[:, :, non_hier_cols].permute((0, 2, 1)).to(device)
        loss = criterion(sigmas_predicted, sigmas_permuted)
        loss.backward()
        total_loss += loss
        optimizer.step()
    print(f'epoch: {epoch}, loss: {total_loss}')

# Save model
path     = f'saved_models/{dataset}/'
filename = "model_prop.pth" if propCycEnc else "model.pth"
filepath = os.path.join(path, filename)
if not os.path.exists(path):
    os.makedirs(path)
torch.save(model.state_dict(), filepath)


/Users/fouadabiad/Projects/silofuse_timeseries/wavestitch/SSSDS4_diffusion.py:94: RuntimeWarning: divide by zero encountered in matmul
  A = T @ M @ np.linalg.inv(T)
/Users/fouadabiad/Projects/silofuse_timeseries/wavestitch/SSSDS4_diffusion.py:94: RuntimeWarning: overflow encountered in matmul
  A = T @ M @ np.linalg.inv(T)
/Users/fouadabiad/Projects/silofuse_timeseries/wavestitch/SSSDS4_diffusion.py:94: RuntimeWarning: invalid value encountered in matmul
  A = T @ M @ np.linalg.inv(T)
/Users/fouadabiad/Projects/silofuse_timeseries/wavestitch/SSSDS4_diffusion.py:94: RuntimeWarning: divide by zero encountered in matmul
  A = T @ M @ np.linalg.inv(T)
/Users/fouadabiad/Projects/silofuse_timeseries/wavestitch/SSSDS4_diffusion.py:94: RuntimeWarning: overflow encountered in matmul
  A = T @ M @ np.linalg.inv(T)
/Users/fouadabiad/Projects/silofuse_timeseries/wavestitch/SSSDS4_diffusion.py:94: RuntimeWarning: invalid value encountered in matmul
  A = T @ M @ np.linalg.inv(T)
/var/folders/p4/ny

epoch: 0, loss: 6.008656978607178
epoch: 1, loss: 5.971344947814941
epoch: 2, loss: 6.001814365386963
epoch: 3, loss: 5.9813995361328125
epoch: 4, loss: 5.957625389099121
epoch: 5, loss: 5.900445938110352
epoch: 6, loss: 5.841930389404297
epoch: 7, loss: 5.745138645172119
epoch: 8, loss: 5.639784812927246
epoch: 9, loss: 5.50783109664917
epoch: 10, loss: 5.340834617614746
epoch: 11, loss: 5.176697254180908
epoch: 12, loss: 4.931811332702637
epoch: 13, loss: 4.725268840789795
epoch: 14, loss: 4.472095489501953
epoch: 15, loss: 4.243905544281006
epoch: 16, loss: 4.007968425750732
epoch: 17, loss: 3.7768473625183105
epoch: 18, loss: 3.557783603668213
epoch: 19, loss: 3.3350062370300293
epoch: 20, loss: 3.156740427017212
epoch: 21, loss: 2.98233699798584
epoch: 22, loss: 2.816852331161499
epoch: 23, loss: 2.681504964828491
epoch: 24, loss: 2.5759425163269043
epoch: 25, loss: 2.480567455291748
epoch: 26, loss: 2.35336971282959
epoch: 27, loss: 2.272810935974121
epoch: 28, loss: 2.1983110904

KeyboardInterrupt: 